In [ ]:
%py
# Purpose: Mask the last four digits of invoice_number in purgo_playground.d_product_revenue_clone table

from pyspark.sql.functions import col, when, expr, length, concat, substring
from pyspark.sql.types import StringType

try:
    # Drop the clone table if it exists
    spark.sql("""
        DROP TABLE IF EXISTS purgo_playground.d_product_revenue_clone
    """)
except Exception as e:
    # Handle failure to drop table
    raise Exception("Failed to drop existing purgo_playground.d_product_revenue_clone table") from e

try:
    # Create a replica of d_product_revenue table
    spark.sql("""
        CREATE TABLE purgo_playground.d_product_revenue_clone AS
        SELECT * FROM purgo_playground.d_product_revenue
    """)
except Exception as e:
    # Handle failure to clone table
    raise Exception("Failed to create purgo_playground.d_product_revenue_clone table") from e

try:
    # Read the cloned table into a DataFrame
    df_clone = spark.table("purgo_playground.d_product_revenue_clone")
    
    # Check if 'invoice_number' column exists
    if 'invoice_number' not in df_clone.columns:
        raise Exception("invoice_number column is missing in purgo_playground.d_product_revenue_clone table")
    
    # Convert 'invoice_number' to string and mask the last four digits
    df_masked = df_clone.withColumn(
        "invoice_number",
        when(
            col("invoice_number").isNotNull(),
            concat(
                substring(col("invoice_number").cast(StringType()), 1, length(col("invoice_number").cast(StringType())) - 4),
                "****"
            )
        ).otherwise(None)
    )
    
    # Validate the number of columns matches the target table's schema
    cloned_columns = set(df_clone.columns)
    masked_columns = set(df_masked.columns)
    if cloned_columns != masked_columns:
        raise Exception("Column count mismatch between original and cloned tables after masking")
    
    # Write the masked DataFrame back to the clone table
    df_masked.write.mode("overwrite").saveAsTable("purgo_playground.d_product_revenue_clone")
    
except Exception as e:
    # Handle errors during masking and writing
    raise Exception("Error occurred during masking of invoice_number") from e

# Masking operation completed successfully